# SolarFragments: reproducing the final submissions

Solar Filament Segmentation Challenge 2026. This notebook regenerates our predictions from the released
checkpoints, using only the competition's test images (one image at a time, no other data and no extra labels).

| | pipeline | stage-1 models | TTA | OOF PQ | public LB | s/image (RTX 4090) |
|---|---|---|---|---|---|---|
| **A** (accuracy) | `runs/w11_all_768b` | 8 x 5 folds | D4 | 0.4667 | 0.40 | 18.7 |
| **B** (efficiency) | `runs/w_s2_44h` | 3 x 5 folds | hflip | 0.4623 | 0.40 | 3.3 |

**A is our final submission; B is its efficient variant.** Both use the same code and the same dataset.
This notebook runs B end to end by default. A (`RUN_A = True`) runs 40 U-Nets with D4 TTA: about 8 min per image
on a Kaggle T4, i.e. ~24 h for the 180 test images, which exceeds Kaggle's 12 h session limit (a full attempt stopped
at 73/180 images). To reproduce A, run the same command on a local GPU with at least 16 GB
(`python infer.py --run runs/w11_all_768b --tta d4`, 56 min on an RTX 4090), or run it here in several sessions
on subsets of the test images (`--images <dir>`).

Pipeline: U-Net ensemble probability maps -> tree of candidate instances from 10 threshold levels -> GBM + rotated-crop
refiner scores -> tree selection and refined outlines. Code: MIT license; training code and logs are in the GitHub repo.

Inputs: the competition data and the `solarfragments-weights` dataset (code, checkpoints, pinned wheels).
Kaggle GPUs (T4/P100) have no bf16, so inference runs in fp16 there. Even on the same GPU, cuDNN's kernel choice makes
masks differ by a few pixels between runs (per-filament IoU >= 0.998 against the submitted files).

In [ ]:
import glob, os, subprocess, sys, time

SRC = os.path.dirname(sorted(glob.glob('/kaggle/input/**/infer.py', recursive=True))[0])
IMAGES = sorted(glob.glob('/kaggle/input/**/test_images', recursive=True))[0]
OUT = '/kaggle/working'
RUN_A = False  # 8 members x 5 folds with D4 TTA: ~24 h on a Kaggle T4, over the 12 h limit (18.7 s/image on an RTX 4090)
RUN_B = True   # 3 members x 5 folds with hflip TTA: ~1 h on a Kaggle T4 (3.3 s/image on an RTX 4090)
print(SRC, IMAGES, len(os.listdir(IMAGES)), 'test images')
print('fp16/fp32 fallback present:', 'amp_forward' in open(os.path.join(SRC, 'src', 'inference.py')).read())

In [ ]:
# Pinned inference dependencies (torch/torchvision from the Kaggle image). --no-deps keeps Kaggle's torch untouched;
# wheels ship in the dataset, so this also works with the internet switched off.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--no-index',
                '--find-links', os.path.join(SRC, 'wheels'), '-r', os.path.join(SRC, 'requirements-infer.txt')], check=True)
subprocess.run([sys.executable, '-c', 'import torch, numpy, sklearn, timm, segmentation_models_pytorch as smp; '
                'print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", '
                'numpy.__version__, sklearn.__version__, timm.__version__, smp.__version__)'], check=True)

In [ ]:
def run(name, run_dir, tta):
    out = os.path.join(OUT, f'submission_{name}.csv')
    cmd = [sys.executable, 'infer.py', '--run', run_dir, '--refiner-dir', 'refiners', '--tta', tta,
           '--cache-dir', 'none', '--images', IMAGES, '--out', out]
    t0 = time.time()
    # Run as a subprocess from the dataset dir: the run configs use repo-relative paths, and the freshly
    # installed numpy/scikit-learn need a new interpreter.
    subprocess.run(cmd, cwd=SRC, check=True, env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'})
    print(f'{name}: {(time.time() - t0) / len(os.listdir(IMAGES)):.1f} s/image')
    subprocess.run([sys.executable, os.path.join('tools', 'submit.py'), out], cwd=SRC, check=True)  # validate only
    return out

if RUN_B:
    run('B_fast_hflip', 'runs/w_s2_44h', 'hflip')
if RUN_A:
    run('A_v11_d4', 'runs/w11_all_768b', 'd4')

In [ ]:
import pandas as pd
for f in sorted(glob.glob(os.path.join(OUT, 'submission_*.csv'))):
    d = pd.read_csv(f)
    print(os.path.basename(f), len(d), 'filaments on', d.filament_id.str.split('_').str[0].nunique(), 'images')